In [1]:
### Use this cell because without it notebook crashed with tmp files
import os
import torch._dynamo

TASK_ROOT = "/mnt/newdata/dpanc/benchmarking/GENA_LM"

os.environ["TMPDIR"] = f"{TASK_ROOT}/cache/tmp"
os.environ["TRITON_CACHE_DIR"] = f"{TASK_ROOT}/cache/triton"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = f"{TASK_ROOT}/cache/torchinductor"

for p in [
    os.environ["TMPDIR"],
    os.environ["TRITON_CACHE_DIR"],
    os.environ["TORCHINDUCTOR_CACHE_DIR"],
]:
    os.makedirs(p, exist_ok=True)

torch._dynamo.config.suppress_errors = True

In [ ]:
# user-configurable variables

GENA_HOME = "/home/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch"
EXPERIMENT_CONFIG = "/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/downstream_tasks/expression_prediction/inference_example/inference.yaml"
CHECKPOINT_PATH = "/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM/models/full_model/pytorch_model.bin"

INFERENCE_DIR = None  # if None, use <GENA_HOME>/downstream_tasks/expression_prediction/inference_example
JSON_DIR = "/mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid812/json_14"
FORWARD_INTERVALS_PATH = "/mnt/newdata/dpanc/benchmarking/data/human.valid.forward.csv"
REVERSE_INTERVALS_PATH = "/mnt/newdata/dpanc/benchmarking/data/human.valid.reverse.csv"

GENOME_PATH = "/mnt/newdata/dpanc/benchmarking/data/hg38.fna"
NUM_BEFORE = 512
TOKEN_LEN_FOR_FETCH = 15

DNA_TOKENIZER = None  # if None, use gen_tokenizer from config
TEXT_TOKENIZER = None  # if None, use text_tokenizer from config
DNA_MAX_SEQ_LEN = None  # if None, use input_seq_len from config
TEXT_MAX_SEQ_LEN = None  # if None, use text_max_seq_len from config

# PREDICTION_MATRIX_CSV = "predicted_expression_matrix.csv"
CORR_METHOD = "pearson"


In [3]:
import sys, os
import torch
import json
from pathlib import Path
from transformers import AutoTokenizer

# hydra imports; not really required if you will hard-code model params in future
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

# set GENALM_HOME environment variable to point to GENA_LM repo root; required to process config files
os.environ["GENALM_HOME"] = GENA_HOME 

sys.path.append(GENA_HOME)
sys.path.append(GENA_HOME+"/GENA_LM")

# import model
from downstream_tasks.expression_prediction.expression_model_final import ExpressionCounts
from downstream_tasks.expression_prediction.expression_dataset_final import ExpressionDataset
from downstream_tasks.expression_prediction.inference_example.inference_input_utils import prepare_inference_inputs_from_intervals

/home/dpanc/benchmarking/GENA_LM/envs/expression_flash/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# we have model parameters and other variables in config files; I made one for inference
experiment_config = EXPERIMENT_CONFIG

experiment_config_path = Path(experiment_config).expanduser().absolute()

with initialize_config_dir(str(experiment_config_path.parents[0])):
	experiment_config = compose(config_name=experiment_config_path.name)

model_kwargs = instantiate(experiment_config["model_kwargs"])

# initialize model
model = ExpressionCounts(**model_kwargs)

/tmp/ipykernel_3279554/842616341.py:6: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize_config_dir(str(experiment_config_path.parents[0])):
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`


Using ModernGENA from /home/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/models/modernbert_large/
missing: 0 []
unexpected: 3 ['decoder.bias', 'head.dense.weight', 'head.norm.weight']
mismatched: []
bert dropouts: {'attention_dropout': 0.1, 'embedding_dropout': 0.1, 'mlp_dropout': 0.1}
qwen dropouts: {'attention_dropout': 0.1}
[desc_model] unfrozen transformer blocks: [24, 25, 26, 27] (total blocks=28)
[desc_model] backbone.norm trainable: True (trainable params=1,024)
[desc_model] trainable params: 62,924,800 / 595,776,512
[desc_model] trainable tensors: 45
  - layers.24.self_attn.q_proj.weight
  - layers.24.self_attn.k_proj.weight
  - layers.24.self_attn.v_proj.weight
  - layers.24.self_attn.o_proj.weight
  - layers.24.self_attn.q_norm.weight
  - layers.24.self_attn.k_norm.weight
  - layers.24.mlp.gate_proj.weight
  - layers.24.mlp.up_proj.weight
  - layers.24.mlp.down_proj.weight
  - layers.24.input_layernorm.weight
  - layers.24.post_attention_layernorm.weight
  - layers.25

In [5]:
# load checkpoint
checkpoint_path = CHECKPOINT_PATH 
model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))

<All keys matched successfully>

In [6]:
# configure path-based inference inputs
# descriptions are loaded from all JSON files in json_dir,
# genes are taken from forward intervals (and optionally reverse intervals)

inference_dir = Path(INFERENCE_DIR) if INFERENCE_DIR is not None else Path(GENA_HOME) / "GENA_LM/downstream_tasks/expression_prediction/inference_example"
data_dir = inference_dir / "data"
json_dir = Path(JSON_DIR) if Path(JSON_DIR).is_absolute() else inference_dir / JSON_DIR
forward_intervals_path = Path(FORWARD_INTERVALS_PATH) if Path(FORWARD_INTERVALS_PATH).is_absolute() else inference_dir / FORWARD_INTERVALS_PATH
reverse_intervals_path = None if REVERSE_INTERVALS_PATH is None else (Path(REVERSE_INTERVALS_PATH) if Path(REVERSE_INTERVALS_PATH).is_absolute() else inference_dir / REVERSE_INTERVALS_PATH)
genome_path = Path(GENOME_PATH) 
num_before = int(NUM_BEFORE) 
token_len_for_fetch = TOKEN_LEN_FOR_FETCH


In [7]:
# helper that prepares descriptions from a folder with JSON files
# and tokenizes interval files exactly with the dataset logic

def prepare_inference_inputs(
	json_dir,
	forward_intervals_path,
	genome_path,
	reverse_intervals_path=None,
	gen_tokenizer=None,
	text_tokenizer_override=None,
	gen_max_seq_len_override=None,
	text_max_seq_len_override=None,
):
	gen_tokenizer_used = gen_tokenizer or dna_tokenizer
	text_tokenizer_used = text_tokenizer_override or text_tokenizer
	gen_max_seq_len_used = gen_max_seq_len_override or dna_max_seq_len
	text_max_seq_len_used = text_max_seq_len_override or text_max_seq_len

	return prepare_inference_inputs_from_intervals(
		json_dir=json_dir,
		forward_intervals_path=forward_intervals_path,
		reverse_intervals_path=reverse_intervals_path,
		genome_path=genome_path,
		gen_tokenizer=gen_tokenizer_used,
		text_tokenizer=text_tokenizer_used,
		gen_max_seq_len=gen_max_seq_len_used,
		text_max_seq_len=text_max_seq_len_used,
		cache_dir=inference_dir,
		num_before=num_before,
		token_len_for_fetch=token_len_for_fetch,
	)


In [8]:
# prepare tokenizers
dna_tokenizer_name = DNA_TOKENIZER or experiment_config["args_params"]["gen_tokenizer"]
text_tokenizer_name = TEXT_TOKENIZER or experiment_config["shared_dataset_params"]["text_tokenizer"]
dna_tokenizer = AutoTokenizer.from_pretrained(dna_tokenizer_name)
text_tokenizer = AutoTokenizer.from_pretrained(text_tokenizer_name, padding_side='left')

dna_max_seq_len = int(DNA_MAX_SEQ_LEN) if DNA_MAX_SEQ_LEN is not None else int(experiment_config["args_params"]["input_seq_len"])
text_max_seq_len = int(TEXT_MAX_SEQ_LEN) if TEXT_MAX_SEQ_LEN is not None else int(experiment_config["shared_dataset_params"]["text_max_seq_len"])

In [9]:
# prepare interval-based inference inputs

prepared_inference = prepare_inference_inputs(
	json_dir=json_dir,
	forward_intervals_path=forward_intervals_path,
	genome_path=genome_path,
	reverse_intervals_path=reverse_intervals_path,
)

genes = prepared_inference["genes"]
experiments = prepared_inference["experiments"]
tokenized_DNA = prepared_inference["tokenized_DNA"]
tokenized_descriptions = prepared_inference["tokenized_descriptions"]

print("Gene token caches:", prepared_inference["gene_cache_paths"])
print("Description token cache:", prepared_inference["description_cache_path"])
print("Input IDs shape:", tokenized_DNA["input_ids"].shape)
tokenized_DNA


Gene token caches: {'forward': '/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/GENA_LM/downstream_tasks/expression_prediction/inference_example/inference_dataset_hash.forward.ed91c4ac73e84cf3.h5', 'reverse': '/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/GENA_LM/downstream_tasks/expression_prediction/inference_example/inference_dataset_hash.reverse.75d764e3a7690768.h5'}
Description token cache: /mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/GENA_LM/downstream_tasks/expression_prediction/inference_example/json_812.b4df14daedb5e50b.Qwen_Qwen3-Embedding-0.6B.510.description.h5
Input IDs shape: torch.Size([3038, 1024])


{'input_ids': tensor([[    1,  1030,    31,  ...,     3,     3,     3],
         [    1,   246,   408,  ...,  2295,  4731,     2],
         [    1, 24971,  1525,  ..., 21461,   208,     2],
         ...,
         [    1,   151,  1622,  ...,    57,   946,     2],
         [    1,   570,    87,  ...,   161,   792,     2],
         [    1,   821,    48,  ...,   693,   240,     2]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]),
 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]])}

In [10]:
# inspect tokenized experiment description

first_experiment = next(iter(experiments))
print("Experiments:", list(experiments.keys()))
print("Input IDs shape:", tokenized_descriptions[first_experiment]["input_ids"].shape)
tokenized_descriptions[first_experiment]


Experiments: ['ENCFF003KHL', 'ENCFF003QOJ', 'ENCFF007QAS', 'ENCFF007UXU', 'ENCFF009MEF', 'ENCFF010UKB', 'ENCFF010XLY', 'ENCFF011DHD', 'ENCFF012XRF', 'ENCFF013HFB', 'ENCFF013YPF', 'ENCFF014BZI', 'ENCFF015GRH', 'ENCFF015QYZ', 'ENCFF015XFX', 'ENCFF016ISI', 'ENCFF016NWL', 'ENCFF016TZA', 'ENCFF020OPI', 'ENCFF020WAT', 'ENCFF021RJG', 'ENCFF021ZXF', 'ENCFF022GUW', 'ENCFF022TTZ', 'ENCFF023CIG', 'ENCFF024KOW', 'ENCFF026RXV', 'ENCFF027FUC', 'ENCFF028IUE', 'ENCFF030ORH', 'ENCFF031BVI', 'ENCFF031DLO', 'ENCFF032FHV', 'ENCFF032LLH', 'ENCFF033KAE', 'ENCFF034BJH', 'ENCFF036PUS', 'ENCFF036XST', 'ENCFF037PQJ', 'ENCFF039WAK', 'ENCFF041WVF', 'ENCFF046TTE', 'ENCFF048JQQ', 'ENCFF050ASS', 'ENCFF050LMR', 'ENCFF051CMQ', 'ENCFF052LZO', 'ENCFF053HTJ', 'ENCFF054JWH', 'ENCFF055STD', 'ENCFF056EWU', 'ENCFF056WDN', 'ENCFF057NDA', 'ENCFF058RCR', 'ENCFF059EED', 'ENCFF061GJD', 'ENCFF064NMJ', 'ENCFF065BDP', 'ENCFF068AID', 'ENCFF068EDD', 'ENCFF068XDC', 'ENCFF068ZME', 'ENCFF069HBW', 'ENCFF069HHK', 'ENCFF071WUF', 'ENCFF074LK

{'input_ids': tensor([[   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643],
         ...,
         [   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]])}

In [11]:
# cast inputs and run forward pass
device = torch.device("cuda:0")

batch_size = 256

# in this example B=len(genes), N=1, L=len(DNA tokens), D=len(text tokens)

input_ids = tokenized_DNA["input_ids"].to(device)  # (B, L)
attention_mask = tokenized_DNA["attention_mask"].to(device)  # (B, L)

model = model.eval()
model.to(device)

outputs = {}

for experiment in experiments:

    # descriptions for all genes
    desc_input_ids_all = tokenized_descriptions[experiment]["input_ids"].to(device)  # (B, D)
    desc_attention_mask_all = tokenized_descriptions[experiment]["attention_mask"].to(device)  # (B, D)

    experiment_outputs = []

    for start in range(0, input_ids.shape[0], batch_size):

        end = min(start + batch_size, input_ids.shape[0])

        # DNA batch
        batch_input_ids = input_ids[start:end]  # (batch_size, L)
        batch_attention_mask = attention_mask[start:end]  # (batch_size, L)

        # Description batch + extra dimension N=1
        batch_desc_input_ids = torch.unsqueeze(
            desc_input_ids_all[start:end], dim=1
        )  # (batch_size, 1, D)

        batch_desc_attention_mask = torch.unsqueeze(
            desc_attention_mask_all[start:end], dim=1
        )  # (batch_size, 1, D)

        # 1 -> repeating DNA; 0 -> repeating DESC
        dataset_flag = torch.zeros(
            size=(batch_input_ids.shape[0], 1),
            device=device,
            dtype=torch.bool,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ), torch.no_grad():

            output = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                desc_input_ids=batch_desc_input_ids,
                desc_attention_mask=batch_desc_attention_mask,
                dataset_flag=dataset_flag,
            )

        experiment_outputs.append(output["logits"][:,0,0])

        if start <= 5 or start % 100 == 0:
            print(f"Processed {start}/{input_ids.shape[0]}")

    outputs[experiment] = experiment_outputs

Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/3038
Processed 0/30

In [ ]:
# build prediction table and gene x cell-type matrix

import pandas as pd

prediction_rows = []
gene_names = list(genes.keys())
cell_type_names = list(experiments.keys())

for cell_type_name in cell_type_names:
	for gene_idx, gene_name in enumerate(gene_names):
		predicted_expression = outputs[cell_type_name]["logits"][gene_idx, 0, 0].item()
		prediction_rows.append(
			{
				"Cell Type": cell_type_name,
				"Gene": gene_name,
				"Predicted Expression": predicted_expression,
			}
		)

predictions_df = pd.DataFrame(prediction_rows)
expression_matrix = predictions_df.pivot(index="Gene", columns="Cell Type", values="Predicted Expression")
expression_matrix = expression_matrix.reindex(index=gene_names, columns=cell_type_names)

display(predictions_df)
display(expression_matrix)

In [12]:
import pandas as pd
gene_tf_data1 = pd.read_csv("/mnt/newdata/dpanc/benchmarking/data/human.valid.forward.csv", sep="\t")
gene_tf_data2 = pd.read_csv("/mnt/newdata/dpanc/benchmarking/data/human.valid.reverse.csv", sep="\t")

gene_tf_data = pd.concat([gene_tf_data1, gene_tf_data2], ignore_index=True)
gene_ids = gene_tf_data["gene_id"].tolist()
gene_ids

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


['ENSG00000232604.1',
 'ENSG00000143942.4',
 'ENSG00000068912.13',
 'ENSG00000170634.12',
 'ENSG00000272156.1',
 'ENSG00000177994.15',
 'ENSG00000115306.15',
 'ENSG00000214595.11',
 'ENSG00000143947.13',
 'ENSG00000163001.11',
 'ENSG00000272180.1',
 'ENSG00000055813.5',
 'ENSG00000028116.17',
 'ENSG00000225226.1',
 'ENSG00000233723.8',
 'ENSG00000285673.1',
 'ENSG00000222030.1',
 'ENSG00000231815.2',
 'ENSG00000285611.1',
 'ENSG00000115421.12',
 'ENSG00000162924.14',
 'ENSG00000162928.8',
 'ENSG00000162929.13',
 'ENSG00000237651.6',
 'ENSG00000173163.10',
 'ENSG00000170340.10',
 'ENSG00000228541.1',
 'ENSG00000115504.14',
 'ENSG00000115507.9',
 'ENSG00000014641.17',
 'ENSG00000169764.15',
 'ENSG00000119862.12',
 'ENSG00000223863.1',
 'ENSG00000119844.15',
 'ENSG00000233694.5',
 'ENSG00000226756.1',
 'ENSG00000115902.10',
 'ENSG00000011523.13',
 'ENSG00000138071.13',
 'ENSG00000281920.1',
 'ENSG00000225815.2',
 'ENSG00000232164.1',
 'ENSG00000143995.19',
 'ENSG00000232046.6',
 'ENSG0000

In [13]:

pred_dict = {}

for cell_type, values in outputs.items():
    # values may be a list of tensors; combine them
    values = torch.cat(values, dim=0).tolist()
    # if isinstance(values, list):
    #     values = torch.cat([v.detach().cpu().flatten() for v in values]).numpy()
    # else:
    #     values = values.detach().cpu().flatten().numpy()

    pred_dict[cell_type] = values

pred_df = pd.DataFrame(pred_dict)
pred_df.insert(0, "gene_id", gene_ids)

pred_df.to_csv("gena_lm_predictions_polina_batch256_numbefore512_cell812.csv", index=False)

pred_df.head()

,gene_id,ENCFF003KHL,ENCFF003QOJ,ENCFF007QAS,ENCFF007UXU,ENCFF009MEF,ENCFF010UKB,ENCFF010XLY,ENCFF011DHD,ENCFF012XRF,...,ENCFF993TGD,ENCFF994ERY,ENCFF994TIH,ENCFF995LYR,ENCFF996NCW,ENCFF996QEJ,ENCFF997ZTN,ENCFF998VKC,ENCFF999JNH,ENCFF999KLY
0,ENSG00000232604.1,-0.003693,0.527344,-0.026245,-0.004944,-0.002731,0.009705,-0.015869,-0.004547,0.010559,...,0.017090,-0.017456,0.015381,-0.015015,-0.009216,-0.006409,0.020630,0.012207,0.044678,-0.013428
1,ENSG00000143942.4,1.312500,1.023438,0.718750,1.421875,1.117188,1.593750,1.265625,1.070312,0.890625,...,1.554688,1.054688,1.679688,1.203125,0.816406,0.953125,1.203125,1.390625,1.398438,1.664062
2,ENSG00000068912.13,2.734375,1.179688,2.703125,2.281250,2.078125,2.390625,2.406250,2.515625,2.593750,...,2.750000,2.625000,2.843750,2.718750,2.718750,2.593750,2.890625,2.906250,2.812500,2.359375
3,ENSG00000170634.12,1.101562,1.625000,1.171875,1.148438,1.031250,1.148438,1.132812,1.195312,0.906250,...,1.867188,1.265625,1.101562,1.109375,0.933594,1.179688,1.132812,1.203125,1.085938,1.265625
4,ENSG00000272156.1,0.008606,0.726562,0.003235,0.011169,0.027344,0.034668,-0.003998,-0.005310,0.013794,...,0.060791,-0.000097,0.039551,-0.002655,0.003357,0.004059,0.026733,0.014771,0.033203,-0.009216


In [14]:
# save gene x cell-type table to CSV

prediction_matrix_csv = Path(PREDICTION_MATRIX_CSV)
if not prediction_matrix_csv.is_absolute():
	prediction_matrix_csv = inference_dir / prediction_matrix_csv

expression_matrix.to_csv(prediction_matrix_csv)
print(f"Saved prediction matrix to: {prediction_matrix_csv}")


Saved prediction matrix to: /home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/inference_example/predicted_expression_matrix.csv
